# 01 — NeighborsMatch: accuracy vs. problem radius

**Main experiment (strong claim).** Alon & Yahav's NeighborsMatch is the canonical over-squashing probe:
vanilla GNNs collapse as the radius grows because information from radius-`r` leaves is squashed through a
single path into the root's fixed-width embedding.

We compare **GCN / GAT / GIN** against our **WalkNet** in two variants:
- **`quotient`** — multi-hop aggregation over the *effective* kQ/I walk operators `M_g[i,j] = dim(e_i·(kQ/I)_g·e_j)`.
- **`walkraw`** — the *same architecture* over the raw operators `A^g` (the architecture-matched ablation).

The **`walkraw` vs `quotient` gap** isolates the kQ/I contribution with everything else held fixed. The claim:
identifying functionally equivalent paths *before* aggregating (so fewer messages are squashed) sustains
accuracy where the raw operator collapses.

> **Why multi-hop (not 1-hop merging):** the multiplicity `n_g(i,j)` is a property of length-`g` *walks*, i.e.
> of the `g`-fold composition of the adjacency — invisible to a single 1-hop layer (a 1-hop in-edge merge under
> mean aggregation is idempotent). The walk operators are precomputed per graph (cached by topology) and
> batched block-diagonally; the heavy path-algebra runs once at data-prep time. On a radius-3 tree the quotient
> already lowers total walk multiplicity by ~30% (see `00_env_check.ipynb`).
>
> **Open question / pivot rule:** whether the multiplicity reduction translates into a real *accuracy* gap at
> larger radius with proper training is the empirical question this notebook answers. If `quotient` does not
> beat `walkraw`/baselines by the day-7 checkpoint, pivot to `03_diagnostic.ipynb` (the diagnostic claim).

In [ ]:
import yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path
from oversquash.data import neighborsmatch_dataset
from oversquash.train import run_neighborsmatch_sweep

CFG = yaml.safe_load(open('../configs/neighborsmatch.yaml'))
RESULTS = Path('../results'); (RESULTS / 'figures').mkdir(parents=True, exist_ok=True); (RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
CFG

## Smoke test (day 1–2): validate the full pipeline at one small radius

Keep it tiny so it runs in seconds. We only want to confirm: data generates, every model trains one sweep
point, and the `quotient`/`walkraw` models consume their block-diagonal walk operators without shape errors.

> Note: at small radius the kQ/I effect is tiny (few redundant paths), so `quotient` and `walkraw` may tie
> here — that is expected. The gap, if any, appears at larger radius. This cell only checks the plumbing.

In [ ]:
smoke = dict(CFG)
smoke['data'] = dict(CFG['data'], radii=[2], n_samples=512)
smoke['train'] = dict(CFG['train'], epochs=20, batch_size=128, patience=10)
smoke_results = run_neighborsmatch_sweep(smoke, neighborsmatch_dataset)
pd.DataFrame(smoke_results)

## Full radius sweep (day 3–4)

The collate for batched walk operators is in place (`collate_walk_operators` + `torch`'s DataLoader). Expect
baselines and `walkraw` to degrade as radius grows; the question is whether `quotient` (effective kQ/I) holds
up longer. Tune `epochs`/`lr`/`hidden_dim` in `configs/neighborsmatch.yaml` for real runs — the defaults are a
starting point, not tuned. NeighborsMatch needs enough capacity and training to separate the models.

In [ ]:
results = run_neighborsmatch_sweep(CFG, neighborsmatch_dataset)
df = pd.DataFrame(results)
df.to_csv(RESULTS / 'tables' / 'neighborsmatch.csv', index=False)
df.pivot(index='radius', columns='model', values='val_acc')

In [ ]:
# Figure: accuracy vs radius, one line per model (the paper's headline figure).
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in df.groupby('model'):
    g = g.sort_values('radius')
    ax.plot(g['radius'], g['val_acc'], marker='o', label=name)
ax.set_xlabel('problem radius $r$'); ax.set_ylabel('accuracy')
ax.set_title('NeighborsMatch: kQ/I vs. baselines')
ax.axhline(0.0, color='grey', lw=0.5); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS / 'figures' / 'neighborsmatch_accuracy_vs_radius.pdf')
fig.savefig(RESULTS / 'figures' / 'neighborsmatch_accuracy_vs_radius.png', dpi=150)
plt.show()